In [3]:
import pandas as pd

# Hugging Face ki auto-converted parquet files read karne ki koshish
url = "https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"

try:
    df = pd.read_parquet(url)
    df.to_csv("unified_phishing_data.csv", index=False)
    print("Success! Shape:", df.shape)
    print(df.head())
except Exception as e:
    print("Parquet path error:", e)

Success! Shape: (61849, 2)
   label                                               text
0      1               https://vpoasss-ne-inbex.gynsujh.cn/
1      1  save 70 percent on super viagra 246663 save 70...
2      1  <!doctypehtml><html class=no-js lang=en><scrip...
3      0                            srhnet.org/Boucher.html
4      0                         nps.gov/history/nr/travel/


In [6]:
import pandas as pd
import re

df = pd.read_csv("unified_phishing_data.csv")

# 1. Basic cleanup & duplicates remove
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].drop_duplicates(subset=["text"]).reset_index(drop=True)

# 2. Content type pehchanna
def detect_type(val):
    if re.search(r"<!doctype\s*html|<html|<body|<script", val, re.I):
        return "html"
    if re.match(r"^(https?://|www\.)\S+$", val) or (
        " " not in val and "." in val and len(val) < 250
    ):
        return "url"
    return "text_or_email"

df["content_type"] = df["text"].apply(detect_type)

# 3. HTML hata kar baqi data rakhna
df_clean = df[df["content_type"] != "html"].copy().reset_index(drop=True)

print("HTML hatane ke baad total rows:", len(df_clean))
print(df_clean["content_type"].value_counts())

df_clean.to_csv("no_html_phishing_data.csv", index=False)

HTML hatane ke baad total rows: 49125
content_type
url              32924
text_or_email    16201
Name: count, dtype: int64


In [7]:
import pandas as pd

df = pd.read_csv("no_html_phishing_data.csv")
df["char_len"] = df["text"].astype(str).str.len()
df["word_count"] = df["text"].astype(str).str.split().str.len()

# 1. Label distribution by content_type (URL vs Text/Email mein 0 aur 1 ka ratio)
print("=== LABEL BREAKDOWN BY CONTENT TYPE ===")
print(pd.crosstab(df["content_type"], df["label"], margins=True))
print("\nPercentage breakdown:")
print(pd.crosstab(df["content_type"], df["label"], normalize="index").round(3))

# 2. Text length & word count stats
print("\n=== CHARACTER LENGTH STATS ===")
print(df.groupby(["content_type", "label"])["char_len"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1))

print("\n=== WORD COUNT STATS ===")
print(df.groupby(["content_type", "label"])["word_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1))

# 3. Sample rows dekhna (Phishing = 1 vs Legitimate = 0)
print("\n=== SAMPLE PHISHING (label=1) ===")
for ctype, grp in df[df["label"] == 1].groupby("content_type"):
    print(f"\n[{ctype.upper()}]")
    for s in grp["text"].head(3):
        print("  -", s[:140])

print("\n=== SAMPLE LEGITIMATE (label=0) ===")
for ctype, grp in df[df["label"] == 0].groupby("content_type"):
    print(f"\n[{ctype.upper()}]")
    for s in grp["text"].head(3):
        print("  -", s[:140])

=== LABEL BREAKDOWN BY CONTENT TYPE ===
label              0      1    All
content_type                      
text_or_email   9980   6221  16201
url            17592  15332  32924
All            27572  21553  49125

Percentage breakdown:
label              0      1
content_type               
text_or_email  0.616  0.384
url            0.534  0.466

=== CHARACTER LENGTH STATS ===
                       count    mean       std  min    25%    50%     75%  \
content_type  label                                                         
text_or_email 0       9980.0  3484.4  170583.4  3.0  295.0  841.0  1846.0   
              1       6221.0  1444.6    3180.9  4.0  264.0  617.0  1385.0   
url           0      17592.0    46.6      24.4  4.0   30.0   41.0    58.0   
              1      15332.0    45.1      45.0  4.0   23.0   32.0    50.0   

                        95%         max  
content_type  label                      
text_or_email 0      5976.2  17036692.0  
              1      5671.0  

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("no_html_phishing_data.csv")
df["text"] = df["text"].astype(str).str.strip()
df["char_len"] = df["text"].str.len()

# 1. Noise & extreme outliers filter karein
# URLs: kam se kam 5 chars, zyada se zyada 300 chars
url_mask = (df["content_type"] == "url") & (df["char_len"].between(5, 300))

# Text/Email: kam se kam 10 chars, zyada se zyada 15,000 chars (giant dumps hata kar)
txt_mask = (df["content_type"] == "text_or_email") & (df["char_len"].between(10, 15000))

df_ready = df[url_mask | txt_mask].drop(columns=["char_len"]).reset_index(drop=True)

print("Cleaning se pehle:", len(df), "| Cleaning ke baad:", len(df_ready))
print("\nCleaned content_type:\n", df_ready["content_type"].value_counts())
print("\nCleaned labels:\n", df_ready["label"].value_counts())

# 2. Stratified 80/20 Train-Test Split
train_df, test_df = train_test_split(
    df_ready,
    test_size=0.2,
    random_state=42,
    stratify=df_ready["label"]
)

train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)
print("\nSaved train_data.csv:", train_df.shape, "| test_data.csv:", test_df.shape)

Cleaning se pehle: 49125 | Cleaning ke baad: 48923

Cleaned content_type:
 content_type
url              32887
text_or_email    16036
Name: count, dtype: int64

Cleaned labels:
 label
0    27473
1    21450
Name: count, dtype: int64

Saved train_data.csv: (39138, 3) | test_data.csv: (9785, 3)


In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Train aur Test datasets load karein
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

# 2. TF-IDF Vectorizer (word + char n-grams URLs aur text dono ke liye mufeed hain)
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=35000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["text"].astype(str))
y_train = train_df["label"]

X_test = vectorizer.transform(test_df["text"].astype(str))
y_test = test_df["label"]

# 3. Baseline Logistic Regression Model train karein
model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train, y_train)

# 4. Overall Evaluation
preds = model.predict(X_test)
print("=== OVERALL TEST RESULTS ===")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds, digits=4))

# 5. Category-wise Evaluation (URL vs Text/Email)
test_eval = test_df.copy()
test_eval["pred"] = preds

for ctype, group in test_eval.groupby("content_type"):
    acc = accuracy_score(group["label"], group["pred"])
    print(f"\n--- {ctype.upper()} (n={len(group)}) | Accuracy: {acc:.4f} ---")
    print(classification_report(group["label"], group["pred"], digits=4))

=== OVERALL TEST RESULTS ===
Accuracy: 0.9105
Confusion Matrix:
 [[4998  497]
 [ 379 3911]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9295    0.9096    0.9194      5495
           1     0.8873    0.9117    0.8993      4290

    accuracy                         0.9105      9785
   macro avg     0.9084    0.9106    0.9094      9785
weighted avg     0.9110    0.9105    0.9106      9785


--- TEXT_OR_EMAIL (n=3271) | Accuracy: 0.9538 ---
              precision    recall  f1-score   support

           0     0.9483    0.9794    0.9636      2042
           1     0.9639    0.9113    0.9368      1229

    accuracy                         0.9538      3271
   macro avg     0.9561    0.9454    0.9502      3271
weighted avg     0.9542    0.9538    0.9536      3271


--- URL (n=6514) | Accuracy: 0.8887 ---
              precision    recall  f1-score   support

           0     0.9174    0.8682    0.8921      3453
           1     0.8598    0

In [10]:
import re
import math
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

def calc_entropy(s):
    if not s:
        return 0.0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def extract_url_features(df):
    feats = []
    for text, ctype in zip(df["text"].astype(str), df["content_type"]):
        if ctype == "url":
            url_len = len(text)
            dots = text.count(".")
            hyphens = text.count("-")
            slashes = text.count("/")
            digits = sum(c.isdigit() for c in text)
            has_https = 1 if text.lower().startswith("https://") else 0
            has_http = 1 if text.lower().startswith("http://") else 0
            has_at = 1 if "@" in text else 0
            has_ip = 1 if re.search(r"(\d{1,3}\.){3}\d{1,3}", text) else 0
            entropy = calc_entropy(text)
            is_url = 1
        else:
            # Non-URL rows ke liye neutral/zero structural URL values
            url_len = dots = hyphens = slashes = digits = 0
            has_https = has_http = has_at = has_ip = 0
            entropy = 0.0
            is_url = 0

        feats.append([
            is_url, url_len, dots, hyphens, slashes,
            digits, has_https, has_http, has_at, has_ip, entropy
        ])
    return np.array(feats, dtype=np.float32)

# 1. TF-IDF features
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(train_df["text"].astype(str))
X_test_tfidf = vectorizer.transform(test_df["text"].astype(str))

# 2. Lexical URL features combine karein
X_train_num = csr_matrix(extract_url_features(train_df))
X_test_num = csr_matrix(extract_url_features(test_df))

X_train = hstack([X_train_tfidf, X_train_num])
X_test = hstack([X_test_tfidf, X_test_num])

y_train = train_df["label"]
y_test = test_df["label"]

# 3. Random Forest Model train karein
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=4,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)

# 4. Evaluation
preds = rf_model.predict(X_test)
print("=== OVERALL TEST RESULTS (TF-IDF + URL FEATURES + RF) ===")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds, digits=4))

test_eval = test_df.copy()
test_eval["pred"] = preds
for ctype, group in test_eval.groupby("content_type"):
    acc = accuracy_score(group["label"], group["pred"])
    print(f"\n--- {ctype.upper()} (n={len(group)}) | Accuracy: {acc:.4f} ---")
    print(classification_report(group["label"], group["pred"], digits=4))

=== OVERALL TEST RESULTS (TF-IDF + URL FEATURES + RF) ===
Accuracy: 0.9513
Confusion Matrix:
 [[5321  174]
 [ 303 3987]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9461    0.9683    0.9571      5495
           1     0.9582    0.9294    0.9436      4290

    accuracy                         0.9513      9785
   macro avg     0.9522    0.9489    0.9503      9785
weighted avg     0.9514    0.9513    0.9512      9785


--- TEXT_OR_EMAIL (n=3271) | Accuracy: 0.9593 ---
              precision    recall  f1-score   support

           0     0.9488    0.9882    0.9681      2042
           1     0.9790    0.9113    0.9440      1229

    accuracy                         0.9593      3271
   macro avg     0.9639    0.9498    0.9560      3271
weighted avg     0.9601    0.9593    0.9590      3271


--- URL (n=6514) | Accuracy: 0.9472 ---
              precision    recall  f1-score   support

           0     0.9445    0.9566    0.9505      3453

In [11]:
# Test set par ghalat predictions dekhne ka chhota sa code
test_eval = test_df.copy()
test_eval["pred"] = preds

false_negatives = test_eval[(test_eval["label"] == 1) & (test_eval["pred"] == 0)]
false_positives = test_eval[(test_eval["label"] == 0) & (test_eval["pred"] == 1)]

print("Missed Phishing (False Negatives):", len(false_negatives))
for t in false_negatives["text"].head(5):
    print("  [FN]:", str(t)[:120])

print("\nFalse Alarms (False Positives):", len(false_positives))
for t in false_positives["text"].head(5):
    print("  [FP]:", str(t)[:120])

Missed Phishing (False Negatives): 303
  [FN]: my-puerto-vallarta.com/form/use/adminn/form1.html
  [FN]: hot xxx ( 917 ) join in on the fun with us ! http : / / 38 . 150 . 73 . 20 / babez / index . html * * * * * * * * * * * 
  [FN]: frederic-guerouet.com/Paypal_Virefication/
  [FN]: banking.co.at:443/appl/ebp/useBankLogin.html?token=vtAVoAdwatrwInliNp4VQgDGeg0c8B0l_QKUmpZgMxtRCySCtfTyNkUEYsJGVBX*osyGc
  [FN]: ph ^ armacy get all yourpqlldy that and each base of the climb . mind thick thick mind base of the climb . each and that

False Alarms (False Positives): 174
  [FP]: mycomicpost.com/?p=1291
  [FP]: suberic.net
  [FP]: http://www.washtimes.com/national/20020817-704732.htmThis
  [FP]: http://urology-textbook.com/proteinuria.html
  [FP]: asmt.mafoi.com/login.cfm?mftypeofuser=M


In [12]:
import re
import math
import joblib
import numpy as np
from scipy.sparse import hstack, csr_matrix

# 1. Model aur Vectorizer save karein (training script ke baad chalayein)
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(rf_model, "phishing_rf_model.joblib")
print("Model aur vectorizer save ho gaye!")

# 2. Inference helper functions
def calc_entropy(s):
    if not s:
        return 0.0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def detect_content_type(val):
    s = str(val).strip()
    if re.match(r"^(https?://|www\.)\S+$", s) or (
        " " not in s and "." in s and len(s) < 300
    ):
        return "url"
    return "text_or_email"

def extract_single_url_features(text, ctype):
    if ctype == "url":
        return np.array([[
            1,
            len(text),
            text.count("."),
            text.count("-"),
            text.count("/"),
            sum(c.isdigit() for c in text),
            1 if text.lower().startswith("https://") else 0,
            1 if text.lower().startswith("http://") else 0,
            1 if "@" in text else 0,
            1 if re.search(r"(\d{1,3}\.){3}\d{1,3}", text) else 0,
            calc_entropy(text)
        ]], dtype=np.float32)
    return np.zeros((1, 11), dtype=np.float32)

def predict_phishing(sample_text, threshold=0.5):
    vec = joblib.load("tfidf_vectorizer.joblib")
    clf = joblib.load("phishing_rf_model.joblib")

    s = str(sample_text).strip()
    ctype = detect_content_type(s)

    X_txt = vec.transform([s])
    X_num = csr_matrix(extract_single_url_features(s, ctype))
    X_full = hstack([X_txt, X_num])

    phishing_prob = float(clf.predict_proba(X_full)[0][1])
    verdict = "PHISHING" if phishing_prob >= threshold else "LEGITIMATE"

    return {
        "content_type": ctype,
        "verdict": verdict,
        "phishing_probability": round(phishing_prob, 4),
        "text_preview": s[:100]
    }

# Test examples
samples = [
    "Urgent: Your bank account is suspended. Click here to verify your details immediately.",
    "http://paypal-security-check-login.com/verify?user=123",
    "Hey, are we still meeting at the library around 3 PM today?"
]

for item in samples:
    print(predict_phishing(item))

Model aur vectorizer save ho gaye!
{'content_type': 'text_or_email', 'verdict': 'PHISHING', 'phishing_probability': 0.9517, 'text_preview': 'Urgent: Your bank account is suspended. Click here to verify your details immediately.'}
{'content_type': 'url', 'verdict': 'PHISHING', 'phishing_probability': 0.783, 'text_preview': 'http://paypal-security-check-login.com/verify?user=123'}
{'content_type': 'text_or_email', 'verdict': 'LEGITIMATE', 'phishing_probability': 0.1129, 'text_preview': 'Hey, are we still meeting at the library around 3 PM today?'}
